# Практика 5. QLoRA для классификации банковских обращений

**Задача:** превратить небольшую instruct-модель в классификатор 77 банковских интентов.

Вход — сообщение клиента: `Why is my cash withdrawal still pending?`  
Выход — ровно одна метка: `pending_cash_withdrawal`.

## 1. LoRA и QLoRA в одном уравнении

Для замороженного слоя $W\in\mathbb{R}^{d_{out}\times d_{in}}$ LoRA обучает низкоранговую поправку:

$$W' = W + \frac{\alpha}{r}BA,$$

где $A\in\mathbb{R}^{r\times d_{in}}$, $B\in\mathbb{R}^{d_{out}\times r}$ и $r\ll d$.

| Режим | Base model | Что обучается |
|---|---|---|
| Full fine-tuning | FP16/BF16 | Все веса |
| LoRA | FP16/BF16 | Только $A$ и $B$ |
| QLoRA | 4-bit NF4 | Только FP16/BF16 $A$ и $B$ |

QLoRA экономит память base weights. Активации, временные буферы и адаптеры не становятся четырёхбитными автоматически.

**Мини-задача:** для слоя $4096\times4096$ и `rank=16` оцените долю параметров LoRA.


In [ ]:
d_in = d_out = 4096
# rank x d_in и d_out x rank — параметры LoRA
rank = 16
full_params = d_in * d_out
lora_params = rank * (d_in + d_out)
print(f"Full matrix: {full_params:,}")
print(f"LoRA A+B:   {lora_params:,}")
print(f"Share:      {100 * lora_params / full_params:.5f}%")


## 2. Окружение

In [ ]:
%pip install -q \
  transformers==5.19.0 trl==1.15.0 peft==0.21.2 \
  datasets==5.1.0 accelerate==1.15.0 bitsandbytes==0.50.2 \
  sentencepiece matplotlib pandas


In [ ]:
import json, platform, random, re, time
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, datasets, peft, trl, bitsandbytes

print("Python:", platform.python_version())
for package in (torch, transformers, datasets, peft, trl, bitsandbytes):
    print(f"{package.__name__:14s} {package.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
    print("VRAM, GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
@dataclass
class Config:
    fast_mode: bool = True
    seed: int = 42
    model_fast: str = "Qwen/Qwen2.5-0.5B-Instruct"
    model_full: str = "Qwen/Qwen2.5-1.5B-Instruct"
    dataset_name: str = "PolyAI/banking77"
    use_qlora: bool = True
    train_examples_per_class_fast: int = 20
    train_examples_per_class_full: int = 30
    eval_examples_per_class_fast: int = 4
    eval_examples_per_class_full: int = 20
    generation_examples_per_class: int = 2
    max_length: int = 512
    max_steps_fast: int = 100
    max_steps_full: int = 500
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.05
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    output_dir: str = "outputs/banking77_qlora"

    @property
    def model_name(self):
        return self.model_fast if self.fast_mode else self.model_full

    @property
    def train_examples_per_class(self):
        return self.train_examples_per_class_fast if self.fast_mode else self.train_examples_per_class_full

    @property
    def eval_examples_per_class(self):
        return self.eval_examples_per_class_fast if self.fast_mode else self.eval_examples_per_class_full

    @property
    def max_steps(self):
        return self.max_steps_fast if self.fast_mode else self.max_steps_full

cfg = Config()
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)
BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print(cfg)
print("Compute dtype:", COMPUTE_DTYPE)
print("Effective batch:", cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps)


## 3. BANKING77: структура задачи

В каждой строке два поля:

- `text` — реплика клиента на английском языке;
- `label` — число, закодированное объектом `ClassLabel`.

Человекочитаемое имя получается через `features["label"].int2str(label)`.

Датасет имеет официальный train/test split.


In [ ]:
from datasets import ClassLabel, load_dataset
from huggingface_hub import hf_hub_download

BANKING77_FILES = {
    "train": "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv",
    "test": "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv",
}
raw = load_dataset("csv", data_files=BANKING77_FILES)

# Read the canonical id → label mapping from the repository metadata.
info_path = hf_hub_download(
    repo_id=cfg.dataset_name,
    filename="dataset_infos.json",
    repo_type="dataset",
)
with open(info_path, encoding="utf-8") as file:
    dataset_info = json.load(file)["default"]
LABEL_NAMES = dataset_info["features"]["label"]["names"]

label_to_id = {name: index for index, name in enumerate(LABEL_NAMES)}
unknown = (set(raw["train"].unique("category")) | set(raw["test"].unique("category"))) - set(LABEL_NAMES)
assert not unknown, f"Unknown categories in CSV: {sorted(unknown)}"

def encode_label(example):
    return {"label": label_to_id[example["category"]]}

raw = raw.map(encode_label, remove_columns=["category"])
raw = raw.cast_column("label", ClassLabel(names=LABEL_NAMES))
label_feature = raw["train"].features["label"]
NUM_LABELS = len(LABEL_NAMES)
print(raw)
print("Train rows:", len(raw["train"]))
print("Test rows: ", len(raw["test"]))
print("Features:  ", raw["train"].features)
print("Classes:   ", NUM_LABELS)
assert NUM_LABELS == 77


### 3.1. Каталог всех меток

Сначала посмотрим на пространство ответов целиком. Названия — часть контракта: `pending_transfer` и `failed_transfer` считаются разными ответами.


In [ ]:
label_catalog = pd.DataFrame({"label_id": range(NUM_LABELS), "label_name": LABEL_NAMES})
display(label_catalog.style.hide(axis="index"))

THEMES = {
    "card": [x for x in LABEL_NAMES if "card" in x],
    "cash/atm": [x for x in LABEL_NAMES if "cash" in x or "atm" in x],
    "transfer": [x for x in LABEL_NAMES if "transfer" in x or "beneficiary" in x],
    "top_up": [x for x in LABEL_NAMES if "top_up" in x or "topping_up" in x],
    "identity": [x for x in LABEL_NAMES if "identity" in x or "verify" in x],
    "exchange": [x for x in LABEL_NAMES if "exchange" in x or "currency" in x],
}
display(pd.DataFrame([{"theme": k, "n_labels": len(v), "labels": ", ".join(v)} for k, v in THEMES.items()]))


### 3.2. Примеры из разных классов

Выбираем по одному примеру из 16 разных интентов.


In [ ]:
rng = np.random.default_rng(cfg.seed)

def rows_with_label_names(dataset):
    frame = dataset.to_pandas()
    frame["label_name"] = frame["label"].map(label_feature.int2str)
    return frame

train_frame = rows_with_label_names(raw["train"])
chosen_labels = rng.choice(LABEL_NAMES, size=16, replace=False)
varied_examples = pd.concat([
    train_frame[train_frame.label_name == name].sample(1, random_state=cfg.seed)
    for name in chosen_labels
])[['text', 'label_name']].reset_index(drop=True)
display(varied_examples)


### 3.3. Контрастные семейства: где модель будет ошибаться

Обращайте внимание не на тему вообще, а на **состояние операции**:

- `pending_*` — операция ещё обрабатывается;
- `failed_*` / `declined_*` — операция не прошла;
- `reverted_*` — сначала прошла, затем была отменена;
- `*_not_recognised` — клиент не узнаёт операцию;
- `*_fee_charged` — спор относится к комиссии.


In [ ]:
FOCUS_PATTERNS = [
    r"pending_(card_payment|cash_withdrawal|transfer|top_up)",
    r"(declined|failed|reverted)_(card_payment|cash_withdrawal|transfer|top_up)",
    r"(card_payment|cash_withdrawal)_not_recognised",
    r".*fee.*|.*charge.*",
]
focus_labels = []
for pattern in FOCUS_PATTERNS:
    focus_labels.extend([name for name in LABEL_NAMES if re.fullmatch(pattern, name)])
focus_labels = list(dict.fromkeys(focus_labels))
focus_examples = pd.concat([
    train_frame[train_frame.label_name == name].sample(2, random_state=cfg.seed)
    for name in focus_labels
])[['text', 'label_name']].sort_values('label_name').reset_index(drop=True)
display(focus_examples)


### 3.4. Аудит качества и баланса

Проверяем пропуски, точные дубликаты, число примеров на класс и пересечение текстов между train/test.


In [ ]:
test_frame = rows_with_label_names(raw["test"])
audit = pd.Series({
    "train_rows": len(train_frame), "test_rows": len(test_frame),
    "train_missing_text": int(train_frame.text.isna().sum()),
    "test_missing_text": int(test_frame.text.isna().sum()),
    "train_duplicate_texts": int(train_frame.text.duplicated().sum()),
    "test_duplicate_texts": int(test_frame.text.duplicated().sum()),
    "train_test_text_overlap": len(set(train_frame.text) & set(test_frame.text)),
})
display(audit.to_frame("value"))

train_counts = train_frame.label_name.value_counts().reindex(LABEL_NAMES)
test_counts = test_frame.label_name.value_counts().reindex(LABEL_NAMES)
counts = pd.DataFrame({"train": train_counts, "test": test_counts})
display(counts.describe().T)
order = counts.sort_values("train").index
fig, axes = plt.subplots(1, 2, figsize=(15, 10), sharey=True)
counts.loc[order, "train"].plot.barh(ax=axes[0], title="Train examples per intent")
counts.loc[order, "test"].plot.barh(ax=axes[1], title="Test examples — same order")
axes[0].set_ylabel("intent"); plt.tight_layout()


### 3.5. Длина сообщений


In [ ]:
for frame in (train_frame, test_frame):
    frame["words"] = frame.text.str.split().str.len()
    frame["chars"] = frame.text.str.len()

display(train_frame.words.describe(percentiles=[.5, .9, .95, .99]).to_frame("words").T)
print("Shortest examples")
display(train_frame.nsmallest(5, "words")[["text", "label_name", "words"]])
print("Longest examples")
display(train_frame.nlargest(5, "words")[["text", "label_name", "words"]])
plt.figure(figsize=(9, 3.5))
plt.hist(train_frame.words, bins=35, alpha=.7, label="train")
plt.hist(test_frame.words, bins=35, alpha=.7, label="test")
plt.xlabel("words per message"); plt.ylabel("examples"); plt.title("BANKING77 message lengths")
plt.legend(); plt.show()


## 4. Стратифицированный subset

Берём одинаковое число примеров каждого интента, а не первые `N` строк.


In [ ]:
def balanced_subset(dataset, examples_per_class, seed):
    labels = np.asarray(dataset["label"])
    local_rng = np.random.default_rng(seed)
    selected = []
    for label_id in range(NUM_LABELS):
        candidates = np.flatnonzero(labels == label_id)
        assert len(candidates) >= examples_per_class
        local_rng.shuffle(candidates)
        selected.extend(candidates[:examples_per_class].tolist())
    local_rng.shuffle(selected)
    return dataset.select(selected)

train_raw = balanced_subset(raw["train"], cfg.train_examples_per_class, cfg.seed)
eval_raw = balanced_subset(raw["test"], cfg.eval_examples_per_class, cfg.seed + 1)
generation_eval_raw = balanced_subset(raw["test"], cfg.generation_examples_per_class, cfg.seed + 2)
print("Train subset:", len(train_raw), Counter(train_raw["label"]).most_common()[:3])
print("Eval subset: ", len(eval_raw), Counter(eval_raw["label"]).most_common()[:3])
print("Generation evaluation:", len(generation_eval_raw))
assert len(set(train_raw["label"])) == NUM_LABELS
assert len(set(eval_raw["label"])) == NUM_LABELS


## 5. Prompt-completion и chat template

Модель получает полный список допустимых меток в system prompt. Это делает zero-shot baseline честным: задача определена до обучения. Completion содержит только имя класса. На prompt-токенах loss не считается, но они остаются контекстом.


In [ ]:
LABEL_LIST = ", ".join(LABEL_NAMES)
SYSTEM_PROMPT = (
    "Classify the banking customer message. Return exactly one allowed intent label "
    "and no explanation. Allowed labels: " + LABEL_LIST
)

def to_conversation(example):
    label_name = label_feature.int2str(example["label"])
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": example["text"]}],
        "completion": [{"role": "assistant", "content": label_name}],
        "text": example["text"], "label_id": example["label"], "label_name": label_name,
    }

train_ds = train_raw.map(to_conversation, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(to_conversation, remove_columns=eval_raw.column_names)
generation_eval_ds = generation_eval_raw.map(to_conversation, remove_columns=generation_eval_raw.column_names)
display(pd.DataFrame({"customer_text": train_ds[:12]["text"], "completion": train_ds[:12]["label_name"]}))
print("Prompt object:\n", train_ds[0]["prompt"])
print("Completion object:\n", train_ds[0]["completion"])


### 5.1. Как chat template превращает объект в токены


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
messages = train_ds[0]["prompt"] + train_ds[0]["completion"]
rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(rendered)

lengths = [len(tokenizer.apply_chat_template(x["prompt"] + x["completion"], tokenize=True, add_generation_prompt=False)) for x in train_ds]
print(pd.Series(lengths).describe(percentiles=[.5, .9, .99]))
print(f"Truncated at {cfg.max_length}: {np.mean(np.array(lengths) > cfg.max_length):.2%}")


## 6. Загрузка 4-bit модели



In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

quantization_config = None
if cfg.use_qlora:
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
def load_training_model():
    loaded = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        device_map={"": 0},
    )
    loaded.config.use_cache = False
    if cfg.use_qlora:
        loaded = prepare_model_for_kbit_training(
            loaded, use_gradient_checkpointing=True
        )
    return loaded

torch.cuda.reset_peak_memory_stats()
model = load_training_model()
print("Loaded:", cfg.model_name)
print(f"Allocated after load: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


## 7. Генерационная оценка классификатора

- **exact-match accuracy** — ответ совпал с gold label;
- **valid-label rate** — ответ входит в множество 77 меток;
- пары `(gold, prediction)` объясняют характер ошибок.

Используем greedy decoding и фиксированный стратифицированный subset test.


In [ ]:
LABEL_SET = set(LABEL_NAMES)

def normalize_prediction(text):
    first = text.strip().splitlines()[0] if text.strip() else ""
    first = first.strip().strip("`'\" .,:;[]()")
    if first in LABEL_SET:
        return first
    found = [label for label in LABEL_NAMES if re.search(rf"(?<!\w){re.escape(label)}(?!\w)", text)]
    return found[0] if len(set(found)) == 1 else first

def generate_labels(current_model, texts, batch_size=16):
    predictions, raw_answers = [], []
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    current_model.eval(); current_model.config.use_cache = True
    try:
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            rendered = [tokenizer.apply_chat_template([
                {"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}
            ], tokenize=False, add_generation_prompt=True) for text in batch_texts]
            inputs = tokenizer(rendered, return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_length).to(current_model.device)
            with torch.inference_mode():
                outputs = current_model.generate(**inputs, max_new_tokens=16, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            decoded = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            raw_answers.extend(x.strip() for x in decoded)
            predictions.extend(normalize_prediction(x) for x in decoded)
    finally:
        tokenizer.padding_side = old_padding_side
        current_model.config.use_cache = False
    return predictions, raw_answers

def evaluate_generation(current_model, dataset):
    started = time.perf_counter()
    predictions, raw_answers = generate_labels(current_model, dataset["text"])
    result = pd.DataFrame({"text": dataset["text"], "gold": dataset["label_name"], "prediction": predictions, "raw_answer": raw_answers})
    result["correct"] = result.gold == result.prediction
    result["valid"] = result.prediction.isin(LABEL_SET)
    return {
        "accuracy": float(result.correct.mean()),
        "valid_label_rate": float(result.valid.mean()),
        "seconds": float(time.perf_counter() - started),
    }, result


In [ ]:
baseline_metrics, baseline_predictions = evaluate_generation(model, generation_eval_ds)
print("ZERO-SHOT:", baseline_metrics)
display(baseline_predictions.head(20))
print("Most common zero-shot outputs")
display(baseline_predictions.prediction.value_counts().head(15).to_frame("count"))


## 8. LoRA-конфигурация и completion-only loss


In [ ]:
import gc
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

if "trainer" in globals():
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    model = load_training_model()

def tokenize_for_causal_lm(example):
    prompt_text = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(
        example["label_name"], add_special_tokens=False
    )["input_ids"] + [tokenizer.eos_token_id]
    input_ids = prompt_ids + completion_ids
    if len(input_ids) > cfg.max_length:
        raise ValueError(
            f"Sequence length {len(input_ids)} > max_length={cfg.max_length}. "
            "Increase cfg.max_length; do not silently remove the user message."
        )
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + completion_ids,
    }

tokenized_train = train_ds.map(
    tokenize_for_causal_lm,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train",
)
tokenized_eval = eval_ds.map(
    tokenize_for_causal_lm,
    remove_columns=eval_ds.column_names,
    desc="Tokenizing validation",
)

class CausalLMCollator:
    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        input_ids, attention_mask, labels = [], [], []
        for item in features:
            pad = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad)
            attention_mask.append(item["attention_mask"] + [0] * pad)
            labels.append(item["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.config.use_cache = False

train_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate, max_steps=cfg.max_steps,
    warmup_steps=max(1, round(cfg.max_steps * cfg.warmup_ratio)),
    lr_scheduler_type="cosine",
    logging_steps=5, eval_strategy="steps", eval_steps=25 if cfg.fast_mode else 100,
    save_strategy="no",
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=not BF16, bf16=BF16,
    optim="paged_adamw_8bit" if cfg.use_qlora else "adamw_torch",
    report_to="none", seed=cfg.seed,
)
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=CausalLMCollator(),
)
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable: {trainable:,} / {total:,} = {100 * trainable / total:.4f}%")
print("Trainer:", type(trainer).__name__)
print("GPU compute capability:", torch.cuda.get_device_capability(0))
trainer.model.print_trainable_parameters()


In [ ]:
prepared = trainer.train_dataset[0]
if "labels" in prepared:
    labels = np.asarray(prepared["labels"])
else:
    labels = next(iter(trainer.get_train_dataloader()))["labels"][0].cpu().numpy()
print("Tokens:", len(labels))
print("Masked prompt tokens:", int((labels == -100).sum()))
print("Supervised completion tokens:", int((labels != -100).sum()))
assert (labels == -100).any(), "Prompt was not masked"
assert (labels != -100).any(), "Completion has no supervised tokens"


In [ ]:
baseline_eval_loss = trainer.evaluate()["eval_loss"]
print(f"Baseline validation loss: {baseline_eval_loss:.4f}")


## 9. Обучение


In [ ]:
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - started
peak_vram_gib = torch.cuda.max_memory_allocated() / 2**30
print(train_result)
print(f"Training time: {training_seconds:.1f} s")
print(f"Peak allocated VRAM: {peak_vram_gib:.2f} GiB")
print(f"Seconds per optimizer step: {training_seconds / cfg.max_steps:.3f}")


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
display(history.tail(10))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
train_log = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
eval_log = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()
if not train_log.empty: train_log.plot(x="step", y="loss", marker="o", ax=axes[0])
if not eval_log.empty: eval_log.plot(x="step", y="eval_loss", marker="o", color="darkorange", ax=axes[1])
axes[0].set_title("Training loss"); axes[1].set_title("Validation loss"); plt.tight_layout()


## 10. Accuracy после QLoRA


In [ ]:
post_eval_loss = trainer.evaluate()["eval_loss"]
tuned_metrics, tuned_predictions = evaluate_generation(trainer.model, generation_eval_ds)
summary = pd.DataFrame([
    {"stage": "zero-shot", "eval_loss": baseline_eval_loss, **baseline_metrics},
    {"stage": "QLoRA", "eval_loss": post_eval_loss, **tuned_metrics},
])
display(summary)
comparison = baseline_predictions[["text", "gold", "prediction"]].rename(columns={"prediction": "before"})
comparison["after"] = tuned_predictions.prediction
comparison["before_correct"] = comparison.before == comparison.gold
comparison["after_correct"] = comparison.after == comparison.gold
display(comparison.head(30))


### 10.1. Error analysis

Accuracy отвечает «сколько», а таблица ошибок — «почему». Отдельно считаем ошибки между допустимыми интентами, invalid outputs, accuracy каждого класса и частые пары `gold → prediction`.


In [ ]:
errors = tuned_predictions[~tuned_predictions.correct].copy()
confusions = errors[errors.valid].groupby(["gold", "prediction"]).size().sort_values(ascending=False).rename("count").reset_index()
per_class = tuned_predictions.groupby("gold").correct.agg(["mean", "sum", "count"]).rename(columns={"mean": "accuracy", "sum": "correct"}).sort_values(["accuracy", "count"])
print("Top confusions"); display(confusions.head(20))
print("Lowest per-class accuracy"); display(per_class.head(20))
print("Invalid outputs"); display(errors[~errors.valid][["text", "gold", "raw_answer"]].head(20))
print("Representative mistakes"); display(errors[["text", "gold", "prediction", "raw_answer"]].head(25))


In [ ]:
hard_labels = per_class.head(12).index.tolist()
hard = tuned_predictions[tuned_predictions.gold.isin(hard_labels)]
matrix = pd.crosstab(hard.gold, hard.prediction).reindex(index=hard_labels, fill_value=0)
if matrix.shape[1] == 0:
    print("No valid-label predictions to plot. Inspect invalid outputs above.")
else:
    fig, ax = plt.subplots(figsize=(max(10, .55 * len(matrix.columns)), 7))
    image = ax.imshow(matrix.values, aspect="auto", cmap="Blues")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
    ax.set(xlabel="predicted", ylabel="gold", title="Confusions for 12 hardest intents")
    fig.colorbar(image, ax=ax, label="count"); plt.tight_layout()


## 11. Сохранение и reload адаптера


In [ ]:
adapter_dir = Path(cfg.output_dir) / "adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
(adapter_dir / "experiment_config.json").write_text(json.dumps({
    **asdict(cfg), "label_names": LABEL_NAMES,
    "baseline_metrics": baseline_metrics, "tuned_metrics": tuned_metrics,
    # "peak_vram_gib": peak_vram_gib,
}, ensure_ascii=False, indent=2), encoding="utf-8")
size_mib = sum(p.stat().st_size for p in adapter_dir.rglob("*") if p.is_file()) / 2**20
print("Saved:", adapter_dir.resolve()); print(f"Directory size: {size_mib:.1f} MiB")
print(sorted(p.name for p in adapter_dir.iterdir()))


In [ ]:
VERIFY_RELOAD = True
if VERIFY_RELOAD:
    import gc
    from peft import PeftModel
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(cfg.model_name, dtype=COMPUTE_DTYPE, quantization_config=quantization_config, device_map={"": 0})
    reloaded = PeftModel.from_pretrained(base, adapter_dir)
    check_predictions, check_raw = generate_labels(reloaded, generation_eval_ds[:4]["text"], batch_size=4)
    display(pd.DataFrame({"text": generation_eval_ds[:4]["text"], "gold": generation_eval_ds[:4]["label_name"], "reloaded_prediction": check_predictions, "raw": check_raw}))
    assert all(check_raw), "Reloaded model returned an empty answer"


## Ссылки

- [BANKING77 dataset card](https://huggingface.co/datasets/PolyAI/banking77)
- [BANKING77 paper](https://arxiv.org/abs/2003.04807)
- [LoRA](https://arxiv.org/abs/2106.09685)
- [QLoRA](https://arxiv.org/abs/2305.14314)
- [PEFT quantization guide](https://huggingface.co/docs/peft/developer_guides/quantization)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
